### Sagemaker Endpoint 배포

In [5]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorchModel
import os

In [111]:
s3 = boto3.client('s3')

bucket_name = "smwu-cv-team-4"
object_key = "2/6710064d110e40f0ad2bf4aab31bc61b/artifacts/best_model_binary_epoch_29.pth"
local_file_path = "./best_model_binary_epoch_29.pth"

s3.download_file(bucket_name, object_key, local_file_path)
print(f"Downloaded model to {local_file_path}")


Downloaded model to ./best_model_binary_epoch_29.pth


In [103]:
%%writefile inference.py
import torch
from torchvision import transforms, models
from PIL import Image
import base64
import json
from io import BytesIO
import os
import torch.nn as nn

# 모델 로드 함수
def model_fn(model_dir):
    model = models.efficientnet_b0(pretrained=False)
    
    model.classifier[1] = nn.Sequential(
        nn.Linear(model.classifier[1].in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 2),  # OK/NG 이진 분류
    )
    model.load_state_dict(torch.load(f"{model_dir}/model.pth", map_location=torch.device("cpu")))
    model.eval()  # 평가 모드
    return model

# 입력 데이터 전처리
def input_fn(request_body, content_type='application/json'):
    if content_type == 'application/json':
        request = json.loads(request_body)
        image_data = base64.b64decode(request['image'])
        image = Image.open(BytesIO(image_data)).convert('RGB')
        transform = transforms.Compose([
            transforms.Resize((640, 640)),
            transforms.ToTensor(),
        ])
        return transform(image).unsqueeze(0)
    else:
        raise ValueError(f"Unsupported content type: {content_type}")

# 추론 수행
def predict_fn(input_data, model):
    with torch.no_grad():
        output = model(input_data)
        probabilities = torch.sigmoid(output).cpu().numpy()
        predicted_class = probabilities.argmax(axis=1)
    return {
        "probabilities": probabilities[0].tolist(),
        "predicted_class": int(predicted_class[0])
    }

# 출력 데이터 후처리
def output_fn(prediction, accept):
    if accept == 'application/json':
        response = {
            "NG": prediction["probabilities"][0],
            "OK": prediction["probabilities"][1],
            "predicted_class": prediction["predicted_class"]
        }
        return json.dumps(response), 'application/json'
    else:
        raise ValueError(f"Unsupported accept type: {accept}")


Writing inference.py


In [117]:
import os
import tarfile
import boto3

# 로컬 파일 경로 설정
local_model_path = "/home/ec2-user/SageMaker/hyunji/best_model_binary_epoch_29.pth"
inference_script_path = "/home/ec2-user/SageMaker/hyunji/inference.py"  # 추론 스크립트 경로
compressed_model_path = "/home/ec2-user/SageMaker/hyunji/model.tar.gz"

# S3 경로 설정
bucket_name = "smwu-cv-team-4"  # S3 버킷 이름
s3_model_key = "models/binary-classifier/model.tar.gz"
s3_model_path = f"s3://{bucket_name}/{s3_model_key}"

# 모델 파일과 추론 스크립트를 TAR.GZ로 압축
with tarfile.open(compressed_model_path, "w:gz") as tar:
    tar.add(local_model_path, arcname="model.pth")  # 모델 파일 추가
    tar.add(inference_script_path, arcname="inference.py")  # 추론 스크립트 추가
print(f"Model and script compressed to: {compressed_model_path}")

# S3에 업로드
s3 = boto3.client("s3")
s3.upload_file(compressed_model_path, bucket_name, s3_model_key)
print(f"Compressed model uploaded to: {s3_model_path}")


Model and script compressed to: /home/ec2-user/SageMaker/hyunji/model.tar.gz
Compressed model uploaded to: s3://smwu-cv-team-4/models/binary-classifier/model.tar.gz


In [11]:
import sagemaker

role = sagemaker.get_execution_role()
print(role)


arn:aws:iam::730335373015:role/shark_deploy


In [119]:
from sagemaker.pytorch import PyTorchModel
import sagemaker

sagemaker_session = sagemaker.Session()

# 모델 정의
pytorch_model = PyTorchModel(
    model_data='s3://smwu-cv-team-4/models/binary-classifier/model.tar.gz',  # 모델 파일 경로
    role='arn:aws:iam::730335373015:role/shark_deploy',
    entry_point='inference.py',  # 위에서 작성한 스크립트
    framework_version='2.2.0',
    py_version='py310',
    sagemaker_session=sagemaker_session,
)

# 엔드포인트 배포
predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
    endpoint_name='binary-endpoint'
)

print(f"Endpoint deployed: {endpoint_name}")


----------!Endpoint deployed: binary-endpoint


### Endpoint 추론

In [122]:
import json
import base64

# 테스트 이미지 읽기
with open("test3.jpg", "rb") as f:
    image_bytes = f.read()

# JSON 데이터 생성
input_data = {"image": base64.b64encode(image_bytes).decode()}

# JSON 형식으로 저장
with open("test_input.json", "w") as f:
    json.dump(input_data, f)

In [123]:
import json
import boto3

# 엔드포인트 호출
client = boto3.client("sagemaker-runtime")
response = client.invoke_endpoint(
    EndpointName="binary-endpoint",
    ContentType="application/json",
    Body=json.dumps(input_data)  # JSON 직렬화된 데이터
)

# 결과 출력
result = json.loads(response["Body"].read().decode())
print(json.dumps(result, indent=4))


{
    "NG": 0.9596312642097473,
    "OK": 0.05621277540922165,
    "predicted_class": 0
}


---------------------------

### Endpoint 삭제

In [ ]:
import sagemaker
from sagemaker import get_execution_role

# SageMaker 세션 설정
sagemaker_session = sagemaker.Session()
endpoint_name = "binary-endpoint"
# 엔드포인트 삭제
sagemaker_session.delete_endpoint(endpoint_name)

sagemaker_session.delete_endpoint_config(endpoint_name)

In [27]:
import sagemaker

sagemaker_session = sagemaker.Session()
endpoints = sagemaker_session.sagemaker_client.list_endpoints(StatusEquals='InService')
print(endpoints)


{'Endpoints': [{'EndpointName': 'diecasting-endpoint', 'EndpointArn': 'arn:aws:sagemaker:us-east-1:730335373015:endpoint/diecasting-endpoint', 'CreationTime': datetime.datetime(2024, 11, 28, 7, 24, 30, 414000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2024, 11, 28, 7, 29, 25, 835000, tzinfo=tzlocal()), 'EndpointStatus': 'InService'}, {'EndpointName': 'smwu-team5-endpoint-ver10', 'EndpointArn': 'arn:aws:sagemaker:us-east-1:730335373015:endpoint/smwu-team5-endpoint-ver10', 'CreationTime': datetime.datetime(2024, 11, 27, 19, 38, 20, 173000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2024, 11, 27, 19, 43, 26, 601000, tzinfo=tzlocal()), 'EndpointStatus': 'InService'}, {'EndpointName': 'pytorch-inference-2024-11-27-15-40-05-947', 'EndpointArn': 'arn:aws:sagemaker:us-east-1:730335373015:endpoint/pytorch-inference-2024-11-27-15-40-05-947', 'CreationTime': datetime.datetime(2024, 11, 27, 15, 40, 6, 591000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(202

### 모델 테스트용 추론 코드

In [74]:
import torch
from torchvision import transforms, models
from PIL import Image
import torch.nn as nn


# 모델 경로 설정
model_path = './model_artifacts/best_model_binary_epoch_29.pth'  # 모델 파일 경로
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.efficientnet_b0(pretrained=True)

# Feature Extraction
for param in model.features.parameters():
    param.requires_grad = False  # Feature extraction을 위한 freezing

# Output layer 수정
model.classifier[1] = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2),  # OK/NG 이진 분류
)

model = model.to(device)
model.load_state_dict(torch.load(model_path))  # 모델 가중치 로드
model.eval()  # 평가 모드로 전환

# 이미지 전처리
def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((640, 640)),  # 이미지 크기 조정 (모델에 맞는 크기)
        transforms.ToTensor(),          # 텐서로 변환
    ])
    image = transform(image).unsqueeze(0)  # 배치 차원 추가 (1, C, H, W)
    return image

# 이미지 경로
image_path = 'test2.jpg'  # 테스트 이미지 경로

# 이미지 전처리
input_image = preprocess_image(image_path).to(device)  # 입력 이미지도 device로 이동

# 추론 수행
with torch.no_grad():  # 그래디언트 계산을 하지 않음
    output = model(input_image)

# 출력 결과 (예시: 이진 분류의 경우)
prediction = torch.sigmoid(output)  # 이진 분류 모델일 경우 sigmoid 적용
print(f"Prediction probabilities: NG: {prediction[0][0].item()}, OK: {prediction[0][1].item()}")


Prediction probabilities: NG: 0.04906410723924637, OK: 0.9740028381347656
